![image_1780997374029.png](./image_1780997374029.png "image_1780997374029.png")

![image_1780997394440.png](./image_1780997394440.png "image_1780997394440.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import Window
# Initialize Spark session
spark = SparkSession.builder.appName("MusicDataFrames").getOrCreate()

# =========================
# Artists DataFrame
# =========================
artists_data = [
    (1, "Ed Sheeran"),
    (2, "Drake"),
    (3, "Adele")
]

artists_df = spark.createDataFrame(artists_data, ["artist_id", "artist_name"])
artists_df.show()

# =========================
# Songs DataFrame
# =========================
songs_data = [
    (101, 1),
    (102, 1),
    (103, 1),
    (104, 2),
    (105, 2),
    (106, 3)
]

songs_df = spark.createDataFrame(songs_data, ["song_id", "artist_id"])
songs_df.show()

# =========================
# Global Song Rank DataFrame
# =========================
global_song_rank_data = [
    ("2023-01-01", 101, 1),
    ("2023-01-01", 102, 3),
    ("2023-01-01", 104, 5),
    ("2023-01-01", 106, 8),
    ("2023-01-02", 101, 2),
    ("2023-01-02", 102, 4),
    ("2023-01-02", 103, 9),
    ("2023-01-02", 104, 6),
    ("2023-01-02", 105, 10),
    ("2023-01-02", 106, 7),
    ("2023-01-03", 101, 1),
    ("2023-01-03", 102, 5),
    ("2023-01-03", 103, 8),
    ("2023-01-03", 104, 3),
    ("2023-01-03", 105, 7),
    ("2023-01-03", 106, 12),
    ("2023-01-04", 101, 2),
    ("2023-01-04", 102, 6),
    ("2023-01-04", 103, 10),
    ("2023-01-04", 104, 1),
    ("2023-01-04", 105, 9),
    ("2023-01-05", 101, 3),
    ("2023-01-05", 102, 7),
    ("2023-01-05", 104, 4),
    ("2023-01-05", 105, 8),
    ("2023-01-06", 101, 4),
    ("2023-01-06", 104, 2)
]

global_song_rank_df = spark.createDataFrame(global_song_rank_data, ["day", "song_id", "rank"])
global_song_rank_df.show()


In [0]:
result_df = (
    artists_df.join(songs_df, on="artist_id")
    .join(global_song_rank_df, on="song_id")
    .filter(global_song_rank_df.rank <= 10)
    .groupBy("artist_name")
    .agg(f.count("*").alias("cnt"))
    .filter(f.col("cnt") >= 10)
    .withColumn("rn", f.dense_rank().over(Window.orderBy(f.col("cnt").desc())))
    .select(f.col("artist_name"), f.col("rn").alias("artist_rank"))
)
display(result_df)